### RAG 품질 평가
- faithfulness: 답변이 컨텍스트에 기반해 사실적으로 답을 했는가?
- answer_relevancy: 질문과 답변이 잘 맞는가?
- context_precision: 가져온 문맥 중에서 필요한 부분이 얼마나 잘 포함됐나?
- context_call: 정답에 필요한 문맥을 얼마나 빠짐없이 가져왔나?

### 품질 평가 단계
1. 테스트 데이터 셋 만들기
2. RAG 구축
3. 평가 
4. 개선 반복

In [1]:
# 1. 문서 로드
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 2. 문서 임베딩
from langchain_openai import OpenAIEmbeddings, ChatOpenAI

### 테스트 데이터셋 만드는 컨셉
1. 페르소나
    1. 데이터 셋에 맞는 페르소나
    2. 내가 넣고 싶은 페르소나
2. 시나리오
    1. 각 청크(docs)를 1개 참고 해서 답변을 만들 것인지
    2. 각 청크(docs)를 여러 개 참고해서 답변을 만들 것인지
3. 평가 요소 가중치 설정
    1. 5 : 2.5 : 2의 기본 가중치
    2. 4 : 3 : 3

In [ ]:
# 문서 로드
pdf_path = "../../data/Sustainability_report_2024_kr.pdf"

loader = PyMuPDFLoader(pdf_path)
docs = loader.load()

print(len(docs))

83


In [6]:
# 문서 청킹
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100,
)

chunks = splitter.split_documents(docs[:20])
print(len(chunks))

48


In [8]:
# 3. 시나리오 설정 및 페르소나 생성
from ragas.llms import LangchainLLMWrapper
from ragas.llms.base import llm_factory
from ragas.embeddings import OpenAIEmbeddings
import openai

gen_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini"))
llm = llm_factory('gpt-4.1-mini')
openai_client = openai.OpenAI()
gen_embeddings = OpenAIEmbeddings(client=openai_client)

C:\Users\user\AppData\Local\Temp\ipykernel_41804\3213367038.py:7: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use the modern LLM providers instead: from ragas.llms.base import llm_factory; llm = llm_factory('gpt-4o-mini') or from ragas.llms.base import instructor_llm_factory; llm = instructor_llm_factory('openai', client=openai_client)
  gen_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini"))


In [9]:
from ragas.testset import TestsetGenerator
generator = TestsetGenerator(
    llm=llm,
    embedding_model = gen_embeddings
)

In [10]:
generator.persona_list

### 자동 생성 페르소나 + 커스텀 페르소나 생성
1. 우선 testset 하나를 만들어야 함
2. 자동 생성 페르소나
3. 커스텀 페르소나를 추가

In [22]:
dataset_test = generator.generate_with_langchain_docs(
    documents=chunks,
    testset_size=1
)

Applying HeadlinesExtractor:   0%|          | 0/39 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/48 [00:00<?, ?it/s]

Applying SummaryExtractor:   0%|          | 0/44 [00:00<?, ?it/s]

Property 'summary' already exists in node '7f8612'. Skipping!
Property 'summary' already exists in node '0e0184'. Skipping!
Property 'summary' already exists in node '3dbe40'. Skipping!
Property 'summary' already exists in node 'f830f6'. Skipping!
Property 'summary' already exists in node 'cade3b'. Skipping!


Applying CustomNodeFilter:   0%|          | 0/72 [00:00<?, ?it/s]

...

Applying NERExtractor:   0%|          | 0/55 [00:00<?, ?it/s]

Applying CosineSimilarityBuilder:   0%|          | 0/1 [00:00<?, ?it/s]

Applying OverlapScoreBuilder:   0%|          | 0/1 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/3 [00:00<?, ?it/s]

In [23]:
generator.persona_list

[Persona(name='Sustainability Program Manager', role_description='Leads and implements corporate sustainability initiatives focused on environmental management, resource circulation, and social responsibility compliance.'),
 Persona(name='Corporate Strategy Analyst', role_description='Analyzes global company operations, financial performance, and organizational values to support strategic decision-making and sustainable growth.'),
 Persona(name='Corporate Sustainability Manager', role_description='Oversees sustainability initiatives focused on social impact, labor practices, and innovation to enhance business competitiveness and stakeholder engagement.')]

In [24]:
test_df = dataset_test.to_pandas()
test_df

,user_input,reference_contexts,reference,synthesizer_name
0,What are the recent developments by the SEC re...,[Principle\nPlanet\nPeople\nCEO 메시지\nMessage f...,The U.S. Securities and Exchange Commission (S...,single_hop_specific_query_synthesizer
1,How does Samsung Electronics integrate 환경 및 사회...,[<1-hop>\n\n마련하고 유관 사업 활동을 분석하였습니다. 이후 각 활동과 관...,Samsung Electronics conducts 환경 및 사회 영향 평가 by ...,multi_hop_abstract_query_synthesizer
2,How does Samsung Electronics engage with 투자자 t...,[<1-hop>\n\n기회 등에 대해 논의하였습니다. 삼성전자는 참석한 이해관계자들...,Samsung Electronics engages with 투자자 by sharin...,multi_hop_specific_query_synthesizer


In [25]:
# 커스텀 페르소나 말들기
from ragas.testset.persona import Persona
custom_personas = [Persona(name='Aspiring Professional', role_description='Actively seeks employment opportunities by evaluating companies based on their corporate culture, career growth prospects, and commitment to social and environmental responsibility to find a stable and value-aligned workplace.'), 
                   Persona(name='Individual Investor', role_description="Makes investment decisions by analyzing a company's financial health, market position, and long-term sustainability, including its ESG (Environmental, Social, and Governance) performance, to maximize financial returns and mitigate risks."), 
                   Persona(name='Supplier/Partner Representative', role_description="Represents a supplier or partner company, focusing on maintaining a stable and collaborative business relationship. Responsible for ensuring operational excellence, quality control, and compliance with the client's supply chain standards, including ethical and sustainability requirements.")]


In [26]:
auto_persona = generator.persona_list
auto_persona

[Persona(name='Sustainability Program Manager', role_description='Leads and implements corporate sustainability initiatives focused on environmental management, resource circulation, and social responsibility compliance.'),
 Persona(name='Corporate Strategy Analyst', role_description='Analyzes global company operations, financial performance, and organizational values to support strategic decision-making and sustainable growth.'),
 Persona(name='Corporate Sustainability Manager', role_description='Oversees sustainability initiatives focused on social impact, labor practices, and innovation to enhance business competitiveness and stakeholder engagement.')]

In [28]:
generator.persona_list = auto_persona + custom_personas
generator.persona_list

[Persona(name='Sustainability Program Manager', role_description='Leads and implements corporate sustainability initiatives focused on environmental management, resource circulation, and social responsibility compliance.'),
 Persona(name='Corporate Strategy Analyst', role_description='Analyzes global company operations, financial performance, and organizational values to support strategic decision-making and sustainable growth.'),
 Persona(name='Corporate Sustainability Manager', role_description='Oversees sustainability initiatives focused on social impact, labor practices, and innovation to enhance business competitiveness and stakeholder engagement.'),
 Persona(name='Aspiring Professional', role_description='Actively seeks employment opportunities by evaluating companies based on their corporate culture, career growth prospects, and commitment to social and environmental responsibility to find a stable and value-aligned workplace.'),
 Persona(name='Individual Investor', role_descrip

In [29]:
from ragas.testset.synthesizers.multi_hop import (
    MultiHopAbstractQuerySynthesizer,
    MultiHopSpecificQuerySynthesizer,
)

from ragas.testset.synthesizers.single_hop.specific import (
    SingleHopSpecificQuerySynthesizer,
)
from ragas.llms.base import llm_factory
ragas_llm = llm_factory(model="gpt-4.1-mini")

scenarios = [
    (SingleHopSpecificQuerySynthesizer(llm=ragas_llm), 0.4),
    (MultiHopAbstractQuerySynthesizer(llm=ragas_llm), 0.3),
    (MultiHopSpecificQuerySynthesizer(llm=ragas_llm), 0.3)
]

In [30]:
dataset = generator.generate(
    testset_size = 100,
    query_distribution = scenarios
)

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/100 [00:00<?, ?it/s]

In [31]:
dataset_df = dataset.to_pandas()
dataset_df

,user_input,reference_contexts,reference,synthesizer_name
0,What Samsung Electronics say about CSR D and E...,[Principle\nPlanet\nPeople\nCEO 메시지\nMessage f...,"Samsung Electronics explains that in 2023, glo...",single_hop_specific_query_synthesizer
1,한종희 삼성전자 부회장 지속가능경영에 대해 알려줘,[있어서는 비제조 분야 및 리스크 분석에 따라 제조 분야 2차 협력회사로 \n근로환...,한종희 삼성전자 부회장은 지속가능경영을 삼성전자가 나아가야 할 방향의 흔들리지 않는...,single_hop_specific_query_synthesizer
2,Could you explain the role of the Device eXper...,[Principle\nPlanet\nPeople\n회사소개\nAbout Us\n삼성...,The Device eXperience (DX) division at Samsung...,single_hop_specific_query_synthesizer
3,삼성닷컴 은 이해관계자 소통에서 어떤 역할을 하나요?,[Facts & Figures \nPrinciple\nPlanet\nPeople\n...,"삼성닷컴은 고객과의 소통 채널 중 하나로, 제품과 서비스 품질, 안전한 제품 사용,...",single_hop_specific_query_synthesizer
4,What is the role of the 글로벌 구매 통합관리 시스템(G-SRM)...,[· 주주·투자자 의견 수렴\n 임직원\n· 안전하고 건강한 근로환경\n· 다양...,The 글로벌 구매 통합관리 시스템(G-SRM) is part of the effo...,single_hop_specific_query_synthesizer
...,...,...,...,...
95,How did the company perform in terms of 폐전자제품 ...,[<1-hop>\n\n사내 폐기물 저감 실천 \n 폐제품 수거 체계 운영 상세내용\...,"In 2021, the company collected 55.9 만 톤 of 폐전자...",multi_hop_specific_query_synthesizer
96,How did Samsung Electronics engage with intern...,[<1-hop>\n\n마련하고 유관 사업 활동을 분석하였습니다. 이후 각 활동과 관...,"In March 2024, Samsung Electronics conducted a...",multi_hop_specific_query_synthesizer
97,How DX부문 achieve 플래티넘 certification for 폐기물 매립...,[<1-hop>\n\n사내 폐기물 저감 실천 \n 폐제품 수거 체계 운영 상세내용\...,DX부문 achieved the 플래티넘 certification for 폐기물 매...,multi_hop_specific_query_synthesizer
98,How DX부문 manage product and manufacturing haza...,[<1-hop>\n\n제품 및 제조과정 우려물질 관리\n∙ 제품 내 우려물질 및 사...,"In 2022년, DX부문 strengthened compliance and man...",multi_hop_specific_query_synthesizer


In [33]:
dataset_df.to_csv("../../data/rag_eval.csv", index=False)